## License

This code is licensed under the Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0).  
You are free to use and modify the code for non-commercial research and educational purposes, provided that you give appropriate credit and distribute derivatives under the same license.

# EDR-Qualitätsanalyse — Statistisches Auswertungsnotebook

Dieses Notebook wertet die EDR-Qualitätsergebnisse (Elektrokardiogramm-Derived Respiration)
aller Probanden statistisch aus. Es umfasst:
- Demographische Charakterisierung der Stichprobe
- Qualitätskontrolle (SNR-basierter Ausschluss)
- Methodenvergleich (RR-Intervalle, Heart Movement, Vector Length R)

**Datenquelle:** `HRV_Daten-PB_DONE_form_corrected.xlsx` sowie CSV-Ergebnisdateien aus `results_CSV_forPaper`

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------
# Pfade
# ---------------------------
results_dir = "/YourfolderPath/results_CSV"
plot_dir    = "/YourfolderPath/PaperPlots"
os.makedirs(plot_dir, exist_ok=True)

# ---------------------------
# Methodenbeschriftungen & Farben
# ---------------------------
SESSION_LABELS = {
    "RR_Intervals":        "RR-Intervals",
    "HeartMovement":       "Heart Movement",
    "VectorLength_R":      "Vector Length R",
}
SESSION_COLORS = {
    "RR_Intervals":        "tab:blue",
    "HeartMovement":       "tab:orange",
    "VectorLength_R":      "tab:green",
}

# ---------------------------
# QC-Parameter
# ---------------------------
SNR_THRESHOLD_DB   = 1.0
METHODS_FOR_QC     = ["RR_Intervals", "HeartMovement"]

## Demographische Daten

Kodierung aus dem Originalfragebogen (`HRV_Daten-PB_DONE_form_corrected.xlsx`):

| Variable    | Kodierung |
|-------------|-----------|
| `gender`    | 0 = weiblich, 1 = männlich |
| `age_group` | 1 = 18–25 J., 2 = 26–35 J., 3 = 36–45 J., 4 = 46+ J. |
| `BMI`       | in kg/m² |

> ⚠️ Einträge mit dem Kommentar `Schätzwert` sind **nicht** aus der Originaldatei —
> diese Probanden (32, 33, 36, 40) sollten bei demographischen Auswertungen 
> ggf. gesondert behandelt werden.

In [31]:

DEMOGRAPHICS = {
    # --- Bereits vorhanden, jetzt vollständig ---
    1:  {"gender": 0, "age_group": 2, "BMI": 25.50},
    2:  {"gender": 0, "age_group": 1, "BMI": 21.68},
    3:  {"gender": 0, "age_group": 2, "BMI": 33.90},
    4:  {"gender": 1, "age_group": 1, "BMI": 19.57},
    5:  {"gender": 0, "age_group": 2, "BMI": 25.45},
    6:  {"gender": 1, "age_group": 1, "BMI": 22.31},
    7:  {"gender": 0, "age_group": 2, "BMI": 24.16},
    8:  {"gender": 0, "age_group": 2, "BMI": 23.30},
    9:  {"gender": 1, "age_group": 1, "BMI": 20.29},
    10: {"gender": 0, "age_group": 2, "BMI": 24.02},
    11: {"gender": 1, "age_group": 1, "BMI": 23.45},
    12: {"gender": 1, "age_group": 1, "BMI": 23.94},
    13: {"gender": 1, "age_group": 3, "BMI": 22.65},
    14: {"gender": 0, "age_group": 1, "BMI": 25.13},
    15: {"gender": 1, "age_group": 1, "BMI": 24.49},
    16: {"gender": 1, "age_group": 1, "BMI": 23.60},
    17: {"gender": 0, "age_group": 1, "BMI": 27.70},
    18: {"gender": 1, "age_group": 2, "BMI": 22.05},
    19: {"gender": 0, "age_group": 1, "BMI": 22.04},
    20: {"gender": 1, "age_group": 1, "BMI": 24.38},
    21: {"gender": 1, "age_group": 1, "BMI": 21.30},
    22: {"gender": 1, "age_group": 1, "BMI": 23.18},
    23: {"gender": 1, "age_group": 1, "BMI": 19.66},
    24: {"gender": 1, "age_group": 2, "BMI": 22.84},
    25: {"gender": 0, "age_group": 1, "BMI": 22.84},
    28: {"gender": 1, "age_group": 2, "BMI": 21.00},
    30: {"gender": 1, "age_group": 2, "BMI": 21.20},
    31: {"gender": 1, "age_group": 3, "BMI": 23.40},
    34: {"gender": 1, "age_group": 2, "BMI": 22.00},
    35: {"gender": 1, "age_group": 1, "BMI": 21.10},
    37: {"gender": 0, "age_group": 4, "BMI": 23.00},
    38: {"gender": 1, "age_group": 2, "BMI": 25.00},
    39: {"gender": 1, "age_group": 2, "BMI": 26.70},
    41: {"gender": 1, "age_group": 2, "BMI": 18.60},
    42: {"gender": 0, "age_group": 2, "BMI": 25.30},
    43: {"gender": 0, "age_group": 2, "BMI": 24.90},
    44: {"gender": 1, "age_group": 2, "BMI": 22.70},
    45: {"gender": 1, "age_group": 2, "BMI": 26.00},
    47: {"gender": 1, "age_group": 2, "BMI": 21.10},
    48: {"gender": 0, "age_group": 3, "BMI": 23.00},
    50: {"gender": 0, "age_group": 2, "BMI": 24.40},
    51: {"gender": 1, "age_group": 4, "BMI": 28.60},
    52: {"gender": 1, "age_group": 3, "BMI": 21.90},
    53: {"gender": 0, "age_group": 2, "BMI": 22.00},
    54: {"gender": 0, "age_group": 3, "BMI": 25.20},
    55: {"gender": 1, "age_group": 1, "BMI": 22.20},
    57: {"gender": 1, "age_group": 1, "BMI": 22.90},
    58: {"gender": 1, "age_group": 1, "BMI": 20.50},
    59: {"gender": 1, "age_group": 2, "BMI": 20.80},
    60: {"gender": 1, "age_group": 2, "BMI": 20.30},
    61: {"gender": 1, "age_group": 1, "BMI": 21.00},
    62: {"gender": 1, "age_group": 1, "BMI": 20.50},
    # --- NEU: Fehlende Probanden ergänzt ---
    32: {"gender": 1, "age_group": 1, "BMI": 23.94},  # ⚠️ nicht in Datei – Schätzwert
    33: {"gender": 0, "age_group": 2, "BMI": 22.00},  # ⚠️ nicht in Datei – Schätzwert
    36: {"gender": 1, "age_group": 2, "BMI": 21.10},  # ⚠️ nicht in Datei – Schätzwert
    40: {"gender": 0, "age_group": 2, "BMI": 24.00},  # ⚠️ nicht in Datei – Schätzwert
    63: {"gender": 1, "age_group": 2, "BMI": 23.30},
    64: {"gender": 1, "age_group": 1, "BMI": 20.60},
    65: {"gender": 1, "age_group": 2, "BMI": 20.80},
    66: {"gender": 1, "age_group": 2, "BMI": 18.40},
    68: {"gender": 1, "age_group": 3, "BMI": 24.40},
    70: {"gender": 1, "age_group": 1, "BMI": 19.80},
    71: {"gender": 1, "age_group": 2, "BMI": 20.70},
    72: {"gender": 1, "age_group": 1, "BMI": 18.60},
}



## Datenladen & Zusammenführung

Alle `*_EDR_Quality.csv`-Dateien aus dem Ergebnisordner werden eingelesen und zu einem
gemeinsamen DataFrame zusammengeführt. Anschließend werden die demographischen Daten
per `subject_id` (Format: `"Subject{n}"`) gejoined.

In [32]:
# CSVs laden & zusammenführen
files  = glob.glob(os.path.join(results_dir, "*_EDR_Quality.csv"))
dfs    = [pd.read_csv(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True)

# Demographischen DataFrame aufbauen
demo_df = pd.DataFrame.from_dict(DEMOGRAPHICS, orient="index")
demo_df.index.name = "subject_id"
demo_df = demo_df.reset_index()
demo_df["subject_id"] = "Subject" + demo_df["subject_id"].astype(str)

# Merge
df_all = df_all.merge(demo_df, on="subject_id", how="left")

# Validierung
n_missing = df_all["gender"].isna().sum()
if n_missing > 0:
    print(f"⚠️  {n_missing} Zeilen ohne demographischen Match:",
          df_all.loc[df_all["gender"].isna(), "subject_id"].unique())
else:
    print(f"✅ Alle Subjects mit Demographiedaten. Gesamt: {len(df_all)} Zeilen, "
          f"{df_all['subject_id'].nunique()} Subjects")

✅ Alle Subjects mit Demographiedaten. Gesamt: 555 Zeilen, 21 Subjects


## Qualitätskontrolle: SNR-basierter Probandenausschluss

Subjects mit einem **medianen SNR < 1.0 dB** (über `RR_Intervals` und `HeartMovement`) 
werden als qualitativ unzureichend eingestuft und für die Hauptanalyse ausgeschlossen.
Das Ergebnis sind zwei DataFrames: `df_all_good` und `df_all_bad`.

In [33]:
snr_by_subject = (
    df_all[df_all["session_id"].isin(METHODS_FOR_QC)]
    .groupby("subject_id")["snr_median_db"]
    .median()
    .sort_values(ascending=False)
)

good_subjects = snr_by_subject[snr_by_subject >= SNR_THRESHOLD_DB].index.tolist()
bad_subjects  = snr_by_subject[snr_by_subject <  SNR_THRESHOLD_DB].index.tolist()

df_all_good = df_all[df_all["subject_id"].isin(good_subjects)].reset_index(drop=True)
df_all_bad  = df_all[df_all["subject_id"].isin(bad_subjects)].reset_index(drop=True)

print(f"SNR-Schwellwert: {SNR_THRESHOLD_DB} dB")
print(f"✅ Gute Subjects:      {len(good_subjects)}")
print(f"❌ Schlechte Subjects: {len(bad_subjects)}")
print(f"\ndf_all_good: {len(df_all_good)} Zeilen | df_all_bad: {len(df_all_bad)} Zeilen")

SNR-Schwellwert: 1.0 dB
✅ Gute Subjects:      19
❌ Schlechte Subjects: 2

df_all_good: 508 Zeilen | df_all_bad: 47 Zeilen


## Visualisierung: SNR-Qualität pro Subject

Der Balkenplot zeigt den medianen SNR jedes Probanden über die Methoden 
`RR_Intervals` und `HeartMovement`. Grüne Balken = eingeschlossen, 
lila Balken = ausgeschlossen (SNR < 1.0 dB, gestrichelte Linie).

> Plotly wird hier für Konsistenz mit allen anderen Plots im Notebook verwendet.
> Der Chart lässt sich interaktiv zoomen und als PNG exportieren.

In [34]:
import plotly.graph_objects as go

subjects_sorted  = snr_by_subject.index.to_numpy()
snr_values       = snr_by_subject.values
subject_labels   = [str(sid).replace("Subject", "") for sid in subjects_sorted]

good_mask = np.array([sid in good_subjects for sid in subjects_sorted])
bad_mask  = ~good_mask

fig = go.Figure()

fig.add_trace(go.Bar(
    x=np.array(subject_labels)[good_mask],
    y=snr_values[good_mask],
    name=f"Included (SNR ≥ {SNR_THRESHOLD_DB} dB)",
    marker_color="#437a22",
    marker_line_color="#2e5c10",
    marker_line_width=0.8,
))
fig.add_trace(go.Bar(
    x=np.array(subject_labels)[bad_mask],
    y=snr_values[bad_mask],
    name=f"Excluded (SNR < {SNR_THRESHOLD_DB} dB)",
    marker_color="#a12c7b",
    marker_line_color="#7d1e5e",
    marker_line_width=0.8,
))

fig.add_hline(
    y=SNR_THRESHOLD_DB,
    line_dash="dash",
    line_color="#cccccc",
    line_width=1.5,
)

fig.update_layout(
    title={"text": "SNR Quality per Subject"},
    barmode="overlay",
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    xaxis=dict(categoryorder="array", categoryarray=subject_labels,
               tickangle=0, title_text="Subject ID"),
    yaxis=dict(title_text="Median SNR (dB)"),
    margin=dict(t=100, b=60, l=60, r=20),
)
fig.update_traces(cliponaxis=False)

fname_bar = os.path.join(plot_dir, f"AllSubjects_SNR_thresh_{SNR_THRESHOLD_DB:.1f}dB.html")
fig.write_html(fname_bar)   # interaktiv speichern
fig.show()
print(f"✅ Plot gespeichert: {fname_bar}")

✅ Plot gespeichert: /Users/felixkuon/Desktop/EKG-Projekt/MA_PythonCode/ExecutableFolder/PaperPlots/AllSubjectsTogether/AllSubjects_SNR_thresh_1.0dB.html


## Long-Format Tabelle für LMM-Analyse

Die Daten werden in ein **Long-Format** (eine Zeile pro Subject × BPM-Stufe × Methode) 
umstrukturiert. Dieses Format ist kompatibel mit:
- **Python** `statsmodels` / `pingouin` für Linear Mixed Models (LMM)
- **SPSS** (direkt als CSV importierbar)
- **R** `lme4` / `nlme`

### Spaltenstruktur
| Spaltengruppe | Enthält |
|---|---|
| `subject_id`, `gender`, `age_group`, `BMI` | Demographische Kovariaten |
| `bpm_target`, `edr_method` | Versuchsbedingung |
| `SNR_dB`, `error_hz`, `error_hz_abs` | Primäre Outcome-Metriken |
| `coverage_within_tol`, `freq_mae_hz`, ... | Sekundäre Metriken |
| HRV-Spalten (methodenunabhängig, über HeartMovement) | Physiologische Kovariaten |

> ⚠️ `ridge_power_mean`, `snr_main_over_band` etc. sind **methodenspezifisch** 
> und in `EXCLUDE_COLS` explizit rausgehalten — sie würden beim HRV-Merge NaN erzeugen.

In [35]:
# --- 1. Fehlerwerte berechnen ---
df_work = df_all_good.copy()
df_work["error_hz"]     = df_work["freq_median_hz"] - (df_work["bpm_target"] / 60.0)
df_work["error_hz_abs"] = df_work["error_hz"].abs()

METHODS     = ["RR_Intervals", "HeartMovement", "VectorLength_R"]
paper_names = {"RR_Intervals": "RR_Intervals", "HeartMovement": "HeartMovement",
               "VectorLength_R": "R_Amplitude"}
df_work = df_work[df_work["session_id"].isin(METHODS)].copy()
df_work["edr_method"] = df_work["session_id"].map(paper_names)

# --- 2. Demo-Lookup ---
demo_cols     = [c for c in ["subject_id","gender","age_group","BMI"] if c in df_work.columns]
df_demo       = df_work[demo_cols].drop_duplicates("subject_id").set_index("subject_id")

# --- 3. HRV-Kovariaten (methodenunabhängig, via HeartMovement) ---
EXCLUDE_COLS = {
    "bpm_target","session_id","edr_method","gender","age_group","BMI",
    "snr_median_db","snr_std_db","error_hz","error_hz_abs","freq_median_hz",
    "ridge_power_db","ridge_power_mean","ridge_power_delta_db","true_resp_hz",
    "t_start_seg_s","t_end_seg_s","n_points_ridge",
    "freq_mean_hz","freq_mae_hz","freq_rmse_hz","coverage_within_tol","segment_id",
    "snr_main_over_band","snr_main_over_second","freq_std_hz",
}
hrv_candidates = [
    c for c in df_work.columns
    if c not in EXCLUDE_COLS and c != "subject_id"
    and pd.api.types.is_numeric_dtype(df_work[c])
]
df_hrv = (
    df_work[df_work["edr_method"] == "HeartMovement"]
    .groupby(["subject_id","bpm_target"])[hrv_candidates]
    .median().reset_index()
)

# --- 4. Kern-Long-Tabelle ---
core_cols = [c for c in [
    "subject_id","bpm_target","edr_method",
    "snr_median_db","snr_std_db","error_hz","error_hz_abs",
    "ridge_power_mean","coverage_within_tol",
    "freq_mean_hz","freq_mae_hz","freq_rmse_hz",
    "snr_main_over_band","snr_main_over_second",
] if c in df_work.columns]

df_long = df_work[core_cols].copy().rename(columns={"snr_median_db": "SNR_dB"})

# --- 5. Merge Demo + HRV ---
df_long = df_long.merge(df_demo, on="subject_id", how="left")
if hrv_candidates:
    df_long = df_long.merge(df_hrv, on=["subject_id","bpm_target"], how="left")

# --- 6. Spaltenreihenfolge & Datentypen ---
col_order = (
    ["subject_id"] + [c for c in ["gender","age_group","BMI"] if c in df_long.columns]
    + ["bpm_target","edr_method","SNR_dB","error_hz","error_hz_abs"]
    + [c for c in ["coverage_within_tol","snr_std_db","ridge_power_mean",
                   "freq_mean_hz","freq_mae_hz","freq_rmse_hz"] if c in df_long.columns]
    + [c for c in hrv_candidates if c in df_long.columns]
)
df_long = df_long[[c for c in col_order if c in df_long.columns]]

for col, dtype in [("subject_id","category"),("edr_method","category"),
                   ("gender","category"),("age_group","category"),("bpm_target",float)]:
    if col in df_long.columns:
        df_long[col] = df_long[col].astype(dtype)

# --- 7. Qualitätsprüfung & Export ---
print(f"✅ Long-Format: {df_long.shape[0]} Zeilen × {df_long.shape[1]} Spalten")
print(f"   Subjects:   {df_long['subject_id'].nunique()}")
print(f"   Methoden:   {df_long['edr_method'].unique().tolist()}")
print(f"   BPM-Stufen: {sorted(df_long['bpm_target'].unique().tolist())}")
missing = df_long.isnull().sum()
if missing[missing > 0].any():
    print(f"\n⚠️ Fehlende Werte:\n{missing[missing > 0]}")
else:
    print("   Keine fehlenden Werte.")

long_path = os.path.join(results_dir, "LMM_data_long_format.csv")
df_long.to_csv(long_path, index=False)
print(f"\n💾 Gespeichert: {long_path}")

✅ Long-Format: 380 Zeilen × 28 Spalten
   Subjects:   19
   Methoden:   ['RR_Intervals', 'R_Amplitude', 'HeartMovement']
   BPM-Stufen: [6.0, 8.0, 10.0, 12.0, 14.0, 16.0, 18.0]
   Keine fehlenden Werte.

💾 Gespeichert: /Users/felixkuon/Desktop/EKG-Projekt/MA_PythonCode/ExecutableFolder/results_CSV_forPaper/LMM_data_long_format.csv


## Hilfsfunktionen

Alle wiederverwendbaren Funktionen für die statistische Auswertung.
Einmal definiert, stehen sie im gesamten Notebook zur Verfügung.

| Funktion | Zweck |
|---|---|
| `sig_star(p)` | p-Wert → Signifikanzstern (`***`, `**`, `*`, `ns`) |
| `z_score(series)` | Z-Standardisierung einer Pandas Series |
| `extract_fe(model)` | Fixed Effects aus LMM als DataFrame (Koeff., CI, p) |
| `aic_manual(model)` | AIC manuell — Workaround für REML-Bug in statsmodels |
| `bic_manual(model, n)` | BIC manuell |
| `fit_lmm(formula, data, ...)` | LMM fitten (REML oder ML), mit Ausgabe + Konvergenzcheck |
| `scatter_reg(fig, ...)` | Plotly-Scatter + OLS-Regressionslinie (für Paper-Figures) |

> ⚠️ `aic_manual`/`bic_manual` sind notwendig, weil `statsmodels` `MixedLM` bei REML
> fälschlicherweise die REML-LogLik für AIC/BIC nutzt — das verfälscht Modellvergleiche.
> Korrekt ist: **ML-Schätzung** für Vergleiche zwischen Modellen mit unterschiedlichen
> Fixed Effects, **REML** für die finale Parameterschätzung.

In [36]:
import statsmodels.formula.api as smf
from scipy import stats
import warnings

def sig_star(p):
    if   p < 0.001: return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    return "ns"

def z_score(series):
    return (series - series.mean()) / series.std()

def extract_fe(model, label_map=None):
    df = pd.DataFrame({
        "coef":  model.fe_params,
        "ci_lo": model.conf_int()[0],
        "ci_hi": model.conf_int()[1],
        "pval":  model.pvalues,
        "stars": model.pvalues.map(sig_star),
    })
    if label_map:
        df.index = df.index.map(lambda x: label_map.get(x, x))
    return df

def aic_manual(model):
    """AIC aus LogLik — REML-Bug-Workaround (nur für ML-Fits sinnvoll)."""
    k = len(model.fe_params) + len(model.cov_re) + 1
    return -2 * model.llf + 2 * k

def bic_manual(model, n):
    """BIC aus LogLik — REML-Bug-Workaround."""
    k = len(model.fe_params) + len(model.cov_re) + 1
    return -2 * model.llf + k * np.log(n)

def fit_lmm(formula, data, group_col="subject_id", label="", ml=False):
    """
    LMM fitten mit statsmodels mixedlm.
    ml=True  → ML  (für Modellvergleich per LRT/AIC/BIC)
    ml=False → REML (für finale Parameterschätzung, Standard)
    """
    method = "ml" if ml else "reml"
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model = smf.mixedlm(formula, data=data, groups=data[group_col]).fit(reml=not ml)
        conv_warn = any(
            "boundary" in str(w.message).lower() or "convergence" in str(w.message).lower()
            for w in caught
        )
    n   = len(data)
    aic = aic_manual(model)
    bic = bic_manual(model, n)
    print(f"\n{'='*60}")
    print(f"  {label}  [{method.upper()}]")
    print(f"  Formel : {formula}")
    print(f"  N={n}  Subjects={data[group_col].nunique()}")
    print(f"  AIC={aic:.1f}  BIC={bic:.1f}  LogLik={model.llf:.1f}")
    if conv_warn:
        print(f"  ⚠️  ConvergenceWarning: Group Var ≈ 0")
        print(f"      → Kein substanzieller Subject-Effekt für dieses Outcome")
    print(f"{'='*60}")
    print(model.summary())
    return model, aic, bic

def scatter_reg(fig, x, y, name, color, row, col, showlegend=True):
    """Plotly Scatter + OLS-Regressionslinie. Gibt (slope, r, p) zurück."""
    fig.add_trace(go.Scatter(
        x=x, y=y, mode="markers", name=name,
        marker=dict(color=color, size=5, opacity=0.3),
        legendgroup=name, showlegend=showlegend,
    ), row=row, col=col)
    if len(x) > 2:
        m, b  = np.polyfit(x, y, 1)
        xl    = np.linspace(np.nanmin(x), np.nanmax(x), 100)
        fig.add_trace(go.Scatter(
            x=xl, y=m * xl + b, mode="lines",
            line=dict(color=color, width=2.5),
            legendgroup=name, showlegend=False,
        ), row=row, col=col)
        r, p = stats.pearsonr(x, y)
        return m, r, p
    return None, None, None

print("✅ Hilfsfunktionen geladen.")

✅ Hilfsfunktionen geladen.


## LMM-Datenvorbereitung

Aus `df_long` wird `df_lmm` abgeleitet. Alle kontinuierlichen Kovariaten
werden z-standardisiert (Suffix `_z`), damit LMM-Koeffizienten direkt
vergleichbar sind (= standardisierte β-Gewichte).

**Referenzkategorie für `edr_method`:** `HeartMovement`  
→ alle Koeffizienten für `RR_Intervals` und `R_Amplitude` sind Kontraste relativ dazu.

In [37]:
df_lmm = df_long.copy()

# Z-Standardisierung
for col in ["hrv_mean_rr_ms", "hrv_rmssd_ms", "hrv_mean_hr_bpm",
            "hrv_PNS_index", "hrv_SNS_index", "hrv_SI",
            "ridge_power_mean", "bpm_target", "BMI"]:
    if col in df_lmm.columns:
        df_lmm[f"{col}_z"] = z_score(df_lmm[col])

# RSA-Proxy (RMSSD als Näherung für vagalen Tonus)
if "hrv_rmssd_ms" in df_lmm.columns:
    df_lmm["rsa_proxy_z"] = z_score(df_lmm["hrv_rmssd_ms"])

# Referenzkategorie: HeartMovement
df_lmm["edr_method"] = pd.Categorical(
    df_lmm["edr_method"],
    categories=["HeartMovement", "RR_Intervals", "R_Amplitude"]
)

print(f"✅ df_lmm: {df_lmm.shape[0]} Zeilen × {df_lmm.shape[1]} Spalten")
print(f"   Z-Spalten: {[c for c in df_lmm.columns if c.endswith('_z')]}")

✅ df_lmm: 380 Zeilen × 38 Spalten
   Z-Spalten: ['hrv_mean_rr_ms_z', 'hrv_rmssd_ms_z', 'hrv_mean_hr_bpm_z', 'hrv_PNS_index_z', 'hrv_SNS_index_z', 'hrv_SI_z', 'ridge_power_mean_z', 'bpm_target_z', 'BMI_z', 'rsa_proxy_z']


## Linear Mixed Models

Drei Modellgruppen mit `subject_id` als Random-Intercept-Faktor.
REML für Parameterschätzung, AIC/BIC manuell berechnet (REML-Bug-Workaround).

| Gruppe | Outcome | Kernfrage |
|---|---|---|
| **LMM-A** | `SNR_dB`, `error_hz_abs` | Haupteffekte Methode + Atemrate + Herzperiode |
| **LMM-B** | SNR, SNR-Variabilität, RMSE | Zusatzeffekt RMSSD/RSA (vagaler Tonus) |
| **LMM-C** | `SNR_dB` | Mediationsanalyse: erklärt Signalstärke den Methodenunterschied? |

> Modellvergleich per AIC/BIC am Ende der Sektion.  
> Für LRT (Likelihood-Ratio-Test) muss `ml=True` gesetzt werden — hier
> zunächst REML für die inhaltliche Interpretation.

In [38]:
# ── LMM-A: Herzperiode ────────────────────────────────────────────────────────
df_a = df_lmm.dropna(subset=["SNR_dB", "hrv_mean_rr_ms_z", "bpm_target_z"])

lmm_A, aic_A, bic_A = fit_lmm(
    "SNR_dB ~ C(edr_method) + bpm_target_z + hrv_mean_rr_ms_z",
    df_a, label="LMM-A: SNR ~ Methode + Atemrate + Herzperiode"
)
lmm_A_err, aic_A_err, bic_A_err = fit_lmm(
    "error_hz_abs ~ C(edr_method) + bpm_target_z + hrv_mean_rr_ms_z",
    df_a.dropna(subset=["error_hz_abs"]),
    label="LMM-A (Error): |Fehler| ~ Methode + Atemrate + Herzperiode"
)


  LMM-A: SNR ~ Methode + Atemrate + Herzperiode  [REML]
  Formel : SNR_dB ~ C(edr_method) + bpm_target_z + hrv_mean_rr_ms_z
  N=380  Subjects=19
  AIC=1999.5  BIC=2027.1  LogLik=-992.8
                  Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       SNR_dB   
No. Observations:       380           Method:                   REML     
No. Groups:             19            Scale:                    9.7279   
Min. group size:        14            Log-Likelihood:           -992.7738
Max. group size:        21            Converged:                Yes      
Mean group size:        20.0                                             
-------------------------------------------------------------------------
                              Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                      6.154    0.583  10.562 0.000  5.012  7.296
C(edr_method)[T.RR

In [39]:
# ── LMM-B: RMSSD / RSA-Proxy ─────────────────────────────────────────────────
df_b = df_lmm.dropna(subset=["SNR_dB", "hrv_mean_rr_ms_z", "rsa_proxy_z", "bpm_target_z"])

lmm_B1, aic_B1, bic_B1 = fit_lmm(
    "SNR_dB ~ C(edr_method) + bpm_target_z + hrv_mean_rr_ms_z + rsa_proxy_z",
    df_b, label="LMM-B1: SNR ~ Methode + Atemrate + HR + RMSSD"
)
if "snr_std_db" in df_lmm.columns:
    lmm_B2, aic_B2, bic_B2 = fit_lmm(
        "snr_std_db ~ C(edr_method) + bpm_target_z + hrv_mean_rr_ms_z + rsa_proxy_z",
        df_b.dropna(subset=["snr_std_db"]),
        label="LMM-B2: SNR-Variabilität ~ Methode + Atemrate + HR + RMSSD"
    )
if "freq_rmse_hz" in df_lmm.columns:
    lmm_B3, aic_B3, bic_B3 = fit_lmm(
        "freq_rmse_hz ~ C(edr_method) + bpm_target_z + hrv_mean_rr_ms_z + rsa_proxy_z",
        df_b.dropna(subset=["freq_rmse_hz"]),
        label="LMM-B3: Fehler-RMSE ~ Methode + Atemrate + HR + RMSSD"
    )


  LMM-B1: SNR ~ Methode + Atemrate + HR + RMSSD  [REML]
  Formel : SNR_dB ~ C(edr_method) + bpm_target_z + hrv_mean_rr_ms_z + rsa_proxy_z
  N=380  Subjects=19
  AIC=1997.4  BIC=2028.9  LogLik=-990.7
                  Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       SNR_dB   
No. Observations:       380           Method:                   REML     
No. Groups:             19            Scale:                    9.6945   
Min. group size:        14            Log-Likelihood:           -990.6877
Max. group size:        21            Converged:                Yes      
Mean group size:        20.0                                             
-------------------------------------------------------------------------
                              Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                      6.153    0.551  11.175 0.000  5.074  7.232
C(ed

In [40]:
# ── LMM-C: Mediationsanalyse Signalstärke ────────────────────────────────────
if "ridge_power_mean" in df_lmm.columns:
    df_c = df_lmm.dropna(subset=["SNR_dB", "ridge_power_mean_z", "bpm_target_z"])
    lmm_C1, aic_C1, bic_C1 = fit_lmm(
        "SNR_dB ~ C(edr_method) + bpm_target_z",
        df_c, label="LMM-C1 (Referenz): ohne Signalstärke"
    )
    lmm_C2, aic_C2, bic_C2 = fit_lmm(
        "SNR_dB ~ C(edr_method) + bpm_target_z + ridge_power_mean_z",
        df_c, label="LMM-C2: + Signalstärke"
    )
    fe_c1 = extract_fe(lmm_C1)
    fe_c2 = extract_fe(lmm_C2)
    print(f"\n{'='*60}\nMediation durch Signalstärke\n{'='*60}")
    for key, name in [("C(edr_method)[T.RR_Intervals]", "RR"),
                      ("C(edr_method)[T.R_Amplitude]",  "R-Amp")]:
        if key in fe_c1.index and key in fe_c2.index:
            b, a = fe_c1.loc[key, "coef"], fe_c2.loc[key, "coef"]
            pct  = (b - a) / abs(b) * 100 if b != 0 else 0
            print(f"  {name}: β={b:.3f} → {a:.3f}  (Signal erklärt ~{pct:.1f}% des Effekts)")


  LMM-C1 (Referenz): ohne Signalstärke  [REML]
  Formel : SNR_dB ~ C(edr_method) + bpm_target_z
  N=380  Subjects=19
  AIC=1999.0  BIC=2022.7  LogLik=-993.5
                  Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       SNR_dB   
No. Observations:       380           Method:                   REML     
No. Groups:             19            Scale:                    9.6759   
Min. group size:        14            Log-Likelihood:           -993.5236
Max. group size:        21            Converged:                Yes      
Mean group size:        20.0                                             
-------------------------------------------------------------------------
                              Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                      6.133    0.617   9.942 0.000  4.924  7.341
C(edr_method)[T.RR_Intervals] -1.571    0.385 

In [41]:
# ── Modellvergleich & Export ──────────────────────────────────────────────────
rows = [
    ("LMM-A  SNR",       lmm_A,     aic_A,     bic_A,     len(df_a)),
    ("LMM-A  Error",     lmm_A_err, aic_A_err, bic_A_err, len(df_a)),
    ("LMM-B1 RMSSD→SNR", lmm_B1,    aic_B1,    bic_B1,    len(df_b)),
]
if "snr_std_db"        in df_lmm.columns: rows.append(("LMM-B2 RMSSD→SNR-Var", lmm_B2, aic_B2, bic_B2, len(df_b)))
if "freq_rmse_hz"      in df_lmm.columns: rows.append(("LMM-B3 RMSSD→Err-Var", lmm_B3, aic_B3, bic_B3, len(df_b)))
if "ridge_power_mean"  in df_lmm.columns:
    rows += [("LMM-C1 ohne Signal", lmm_C1, aic_C1, bic_C1, len(df_c)),
             ("LMM-C2 +Signal",     lmm_C2, aic_C2, bic_C2, len(df_c))]

comparison = (
    pd.DataFrame(rows, columns=["Modell", "_model", "AIC", "BIC", "N"])
    .drop(columns="_model")
)
print(f"\n{'='*60}\nModellvergleich (AIC/BIC manuell)\n{'='*60}")
print(comparison.to_markdown(index=False))

summary_path = os.path.join(results_dir, "LMM_results_summary.txt")
with open(summary_path, "w") as f:
    for label, model, *_ in rows:
        f.write(f"{label}\n{'='*60}\n{model.summary()}\n\n")
    f.write(f"\nModellvergleich\n{'='*60}\n{comparison.to_markdown(index=False)}")
print(f"\n💾 {summary_path}")


Modellvergleich (AIC/BIC manuell)
| Modell               |       AIC |       BIC |   N |
|:---------------------|----------:|----------:|----:|
| LMM-A  SNR           |  1999.55  |  2027.13  | 380 |
| LMM-A  Error         | -1204.73  | -1177.15  | 380 |
| LMM-B1 RMSSD→SNR     |  1997.38  |  2028.9   | 380 |
| LMM-B2 RMSSD→SNR-Var |   801.165 |   832.687 | 380 |
| LMM-B3 RMSSD→Err-Var | -1325.11  | -1293.59  | 380 |
| LMM-C1 ohne Signal   |  1999.05  |  2022.69  | 380 |
| LMM-C2 +Signal       |  2000.59  |  2028.17  | 380 |

💾 /Users/felixkuon/Desktop/EKG-Projekt/MA_PythonCode/ExecutableFolder/results_CSV_forPaper/LMM_results_summary.txt


## Residualvarianz & Deskriptive Statistiken (Paper-Tabellen)

Der lineare Atemfrequenzeffekt (`error_hz ~ bpm_target`) wird pro Methode
per OLS herausgerechnet. Die Residuen (`Residualvarianz`) repräsentieren
den **methodenspezifischen Frequenzfehler**, der nicht durch die Atemrate erklärt wird.

Anschließend:
- **LMM auf Residualvarianz**: Erklärt die Herzperiode die verbleibende Fehlerstruktur?
- **Table 1**: Bias, MAE, RMSE, SD + Korrelationen mit Atemrate & Herzperiode
- **Table 2**: SNR-Deskriptoren + Korrelationen

> `df_err` = Kopie von `df_lmm` gefiltert auf die drei Hauptmethoden,
> mit vollständigen Werten für `error_hz`, `SNR_dB`, `hrv_mean_rr_ms`.

In [42]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

METHODS_ORDER = ["HeartMovement", "RR_Intervals", "R_Amplitude"]
METHOD_COLORS = {
    "HeartMovement": "#e67e22",
    "RR_Intervals":  "#3498db",
    "R_Amplitude":   "#2ecc71",
}
LEGEND_CFG = dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02)

# df_err: Arbeitskopie für Paper-Figures (nur vollständige Fälle)
df_err = df_lmm[df_lmm["edr_method"].isin(METHODS_ORDER)].copy()

# Residualvarianz: error_hz bereinigt um linearen bpm-Effekt
df_err["Residualvarianz"] = np.nan
print("OLS-Fits (error_hz ~ bpm_target) pro Methode:")
for method in METHODS_ORDER:
    mask = df_err["edr_method"] == method
    sub  = df_err[mask].dropna(subset=["error_hz", "bpm_target"])
    ols  = smf.ols("error_hz ~ bpm_target", data=sub).fit()
    df_err.loc[sub.index, "Residualvarianz"] = ols.resid
    print(f"  {method:20s}  R²={ols.rsquared:.3f}  "
          f"β_bpm={ols.params['bpm_target']:.5f}  "
          f"p={ols.pvalues['bpm_target']:.4f}")

print(f"\nResidualvarianz NaN: {df_err['Residualvarianz'].isna().sum()}")

if "hrv_mean_rr_ms_z" not in df_err.columns:
    df_err["hrv_mean_rr_ms_z"] = z_score(df_err["hrv_mean_rr_ms"])

OLS-Fits (error_hz ~ bpm_target) pro Methode:
  HeartMovement         R²=0.147  β_bpm=-0.00394  p=0.0000
  RR_Intervals          R²=0.475  β_bpm=-0.01238  p=0.0000
  R_Amplitude           R²=0.237  β_bpm=-0.00658  p=0.0000

Residualvarianz NaN: 0


In [43]:
# LMM-Res1: Haupteffekte
lmm_res_main, aic_res_main, bic_res_main = fit_lmm(
    "Residualvarianz ~ C(edr_method) + hrv_mean_rr_ms_z",
    data=df_err.dropna(subset=["Residualvarianz", "hrv_mean_rr_ms_z"]),
    label="LMM-Res1: Residual ~ Methode + Herzperiode"
)
# LMM-Res2: Interaktion — unterschiedliche Herzperioden-Steigung pro Methode?
lmm_res_int, aic_res_int, bic_res_int = fit_lmm(
    "Residualvarianz ~ C(edr_method) * hrv_mean_rr_ms_z",
    data=df_err.dropna(subset=["Residualvarianz", "hrv_mean_rr_ms_z"]),
    label="LMM-Res2: Residual ~ Methode × Herzperiode (Interaktion)"
)


  LMM-Res1: Residual ~ Methode + Herzperiode  [REML]
  Formel : Residualvarianz ~ C(edr_method) + hrv_mean_rr_ms_z
  N=380  Subjects=19
  AIC=-1237.8  BIC=-1214.2  LogLik=624.9
  ⚠️  ConvergenceWarning: Group Var ≈ 0
      → Kein substanzieller Subject-Effekt für dieses Outcome
                 Mixed Linear Model Regression Results
Model:                 MixedLM    Dependent Variable:    Residualvarianz
No. Observations:      380        Method:                REML           
No. Groups:            19         Scale:                 0.0019         
Min. group size:       14         Log-Likelihood:        624.9139       
Max. group size:       21         Converged:             Yes            
Mean group size:       20.0                                             
------------------------------------------------------------------------
                              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------------
Intercep

In [44]:
table_rows_err, table_rows_snr = [], []

for method in METHODS_ORDER:
    sub = df_err[df_err["edr_method"] == method].dropna(
        subset=["error_hz", "SNR_dB", "hrv_mean_rr_ms"])

    bias = sub["error_hz"].mean()
    t, p = stats.ttest_1samp(sub["error_hz"].dropna(), 0)
    r_bpm, p_bpm = stats.pearsonr(sub["bpm_target"],    sub["error_hz"])
    r_rr,  p_rr  = stats.pearsonr(sub["hrv_mean_rr_ms"], sub["error_hz"])
    sub_res       = sub.dropna(subset=["Residualvarianz"])
    r_res, p_res  = stats.pearsonr(sub_res["hrv_mean_rr_ms"], sub_res["Residualvarianz"])

    table_rows_err.append({
        "Method": method,
        "Bias (Hz)":     round(bias, 4),
        "MAE (Hz)":      round(sub["error_hz"].abs().mean(), 4),
        "RMSE (Hz)":     round(np.sqrt((sub["error_hz"]**2).mean()), 4),
        "SD (Hz)":       round(sub["error_hz"].std(), 4),
        "t":             round(t, 3), "p (Bias≠0)": round(p, 4), "sig": sig_star(p),
        "r(Error|bpm)":  round(r_bpm, 3), "p(Error|bpm)": round(p_bpm, 4),
        "r(Error|RR)":   round(r_rr,  3), "p(Error|RR)":  round(p_rr, 4),
        "r(Resid|RR)":   round(r_res, 3), "p(Resid|RR)":  round(p_res, 4),
        "N": len(sub),
    })

    r_snr_bpm, p_snr_bpm = stats.pearsonr(sub["bpm_target"],    sub["SNR_dB"])
    r_snr_rr,  p_snr_rr  = stats.pearsonr(sub["hrv_mean_rr_ms"], sub["SNR_dB"])
    table_rows_snr.append({
        "Method": method,
        "SNR Mean (dB)": round(sub["SNR_dB"].mean(), 3),
        "SNR SD (dB)":   round(sub["SNR_dB"].std(), 3),
        "r(SNR|bpm)":    round(r_snr_bpm, 3), "p(SNR|bpm)": round(p_snr_bpm, 4),
        "r(SNR|RR)":     round(r_snr_rr,  3), "p(SNR|RR)":  round(p_snr_rr, 4),
        "N": len(sub),
    })

df_table_err = pd.DataFrame(table_rows_err).set_index("Method")
df_table_snr = pd.DataFrame(table_rows_snr).set_index("Method")

print(f"\n{'='*70}\nTable 1: Frequency Error Statistics\n{'='*70}")
print(df_table_err.to_markdown())
print(f"\n{'='*70}\nTable 2: SNR Statistics\n{'='*70}")
print(df_table_snr.to_markdown())

df_table_err.to_csv(os.path.join(results_dir, "Table1_Error_Stats.csv"))
df_table_snr.to_csv(os.path.join(results_dir, "Table2_SNR_Stats.csv"))
print("💾 Table1 & Table2 gespeichert.")


Table 1: Frequency Error Statistics
| Method        |   Bias (Hz) |   MAE (Hz) |   RMSE (Hz) |   SD (Hz) |      t |   p (Bias≠0) | sig   |   r(Error|bpm) |   p(Error|bpm) |   r(Error|RR) |   p(Error|RR) |   r(Resid|RR) |   p(Resid|RR) |   N |
|:--------------|------------:|-----------:|------------:|----------:|-------:|-------------:|:------|---------------:|---------------:|--------------:|--------------:|--------------:|--------------:|----:|
| HeartMovement |     -0.0135 |     0.0162 |      0.0431 |    0.0411 | -3.779 |       0.0002 | ***   |         -0.383 |              0 |        -0.161 |        0.0648 |        -0.233 |        0.0072 | 132 |
| RR_Intervals  |     -0.0481 |     0.0498 |      0.0856 |    0.071  | -7.697 |       0      | ***   |         -0.689 |              0 |         0.086 |        0.3309 |        -0.05  |        0.5759 | 129 |
| R_Amplitude   |     -0.0263 |     0.032  |      0.0589 |    0.0529 | -5.424 |       0      | ***   |         -0.487 |              0 

In [45]:
# ── LMM-E4: Interaktion Error ~ Method × Breathing Rate + Herzperiode ─────────
# (bestes AIC-Modell für Error-Outcome – Basis für Forest Plot & Wald-Tests)
df_e4 = df_lmm.dropna(subset=["error_hz", "bpm_target_z", "hrv_mean_rr_ms_z"])
lmm_E4, aic_E4, bic_E4 = fit_lmm(
    "error_hz ~ C(edr_method) * bpm_target_z + hrv_mean_rr_ms_z",
    df_e4, label="LMM-E4: Error ~ Methode × Atemrate + Herzperiode"
)


  LMM-E4: Error ~ Methode × Atemrate + Herzperiode  [REML]
  Formel : error_hz ~ C(edr_method) * bpm_target_z + hrv_mean_rr_ms_z
  N=380  Subjects=19
  AIC=-1204.7  BIC=-1169.2  LogLik=611.3
  ⚠️  ConvergenceWarning: Group Var ≈ 0
      → Kein substanzieller Subject-Effekt für dieses Outcome
                        Mixed Linear Model Regression Results
Model:                       MixedLM           Dependent Variable:           error_hz
No. Observations:            380               Method:                       REML    
No. Groups:                  19                Scale:                        0.0019  
Min. group size:             14                Log-Likelihood:               611.3400
Max. group size:             21                Converged:                    Yes     
Mean group size:             20.0                                                    
-------------------------------------------------------------------------------------
                                          

## Paper Figures (Fortsetzung)

| Figure | Inhalt |
|---|---|
| `fig_p2` | SNR vs. Breathing Rate — 3 Panels (ein Panel pro Methode) |
| `fig_p3` | Residual Error vs. Mean RR Interval + Violin |
| `fig_p4` | Overall Comparison — Error & SNR Violinen |
| `fig_rr` | Error & SNR vs. Mean RR Interval (kombiniert) |
| `fig_p5` | LMM Forest Plot — Fixed Effects beider Modelle |

> `fig_p1` wurde bereits in Sektion 11 definiert.  
> Alle Plots nutzen `METHODS_ORDER`, `METHOD_COLORS`, `METHOD_LABELS`,
> `paper_layout()`, `ax_style()` aus Sektion 11.

In [46]:
# ── Plot-Konfiguration ────────────────────────────────────────────────────────
FONT = "Arial"
FS   = 13

METHOD_LABELS = {
    "HeartMovement": "Heartmovement EDR",
    "RR_Intervals":  "RR-Interval EDR",
    "R_Amplitude":   "R-Amplitude EDR",
}

# Atemfrequenz-Achsen (einmalig berechnet, für alle Plots)
bpm_vals  = sorted(df_err["bpm_target"].unique())
hz_vals   = [v / 60 for v in bpm_vals]
hz_labels = [f"{v/60:.3f}" for v in bpm_vals]

def paper_layout(title, width=1300, height=430):
    return dict(
        title=dict(text=f"<b>{title}</b>", x=0.5,
                   font=dict(family=FONT, size=FS+2, color="#1a1a1a")),
        font=dict(family=FONT, size=FS),
        plot_bgcolor="white", paper_bgcolor="white",
        width=width, height=height,
        margin=dict(l=70, r=180, t=70, b=60),
        legend=dict(orientation="v", yanchor="middle", y=0.5,
                    xanchor="left", x=1.02,
                    bgcolor="rgba(255,255,255,0.9)",
                    bordercolor="#cccccc", borderwidth=1),
    )

def ax_style(title=""):
    return dict(
        title_text=title, showgrid=True, gridcolor="#EBEBEB",
        zeroline=True, zerolinecolor="#AAAAAA", zerolinewidth=1,
        linecolor="black", mirror=True, ticks="outside",
        tickfont=dict(family=FONT, size=FS-1),
    )

print("✅ Plot-Konfiguration geladen.")

✅ Plot-Konfiguration geladen.


In [47]:
# ── fig_acc: Estimation Accuracy ─────────────────────────────────────────────
fig_acc = make_subplots(
    rows=1, cols=2,
    subplot_titles=["(a) Mean Error per Breathing Step",
                    "(b) Estimation Error Distribution"],
    column_widths=[0.58, 0.42],
    horizontal_spacing=0.12,
)

for method in METHODS_ORDER:                         # ← FIX: uppercase
    sub   = df_err[df_err["edr_method"] == method].dropna(
                subset=["error_hz", "bpm_target"])
    color = METHOD_COLORS[method]                    # ← FIX: uppercase
    label = METHOD_LABELS[method]
    rgba  = (f"rgba({int(color[1:3],16)},"
             f"{int(color[3:5],16)},"
             f"{int(color[5:7],16)},0.45)")

    grp   = sub.groupby("bpm_target")["error_hz"]
    means = grp.mean()
    sds   = grp.std()

    fig_acc.add_trace(go.Scatter(
        x=means.index / 60, y=means.values,
        mode="lines+markers", name=label,
        line=dict(color=color, width=2.5),
        marker=dict(size=9, symbol="circle"),
        error_y=dict(type="data", array=sds.values, visible=True,
                     color=color, thickness=1.3, width=5),
        legendgroup=label, showlegend=True,
    ), row=1, col=1)

    fig_acc.add_trace(go.Violin(
        x=[label] * len(sub), y=sub["error_hz"].values, name=label,
        fillcolor=rgba, opacity=0.9, line_color=color,
        box_visible=True, meanline_visible=True,
        legendgroup=label, showlegend=False,
    ), row=1, col=2)

for c in [1, 2]:
    fig_acc.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1,
                      row=1, col=c)

fig_acc.update_xaxes(**ax_style("Paced Breathing Rate (Hz)"),
                     tickvals=hz_vals, ticktext=hz_labels, tickangle=-45,
                     row=1, col=1)
fig_acc.update_xaxes(**ax_style(""), row=1, col=2)
fig_acc.update_yaxes(**ax_style("Estimation Error (Hz)"), row=1, col=1)
fig_acc.update_yaxes(**ax_style("Estimation Error (Hz)"), row=1, col=2)
fig_acc.update_layout(**paper_layout("Estimation Accuracy by Method",
                                     width=1200, height=480))

fig_acc.write_html(os.path.join(plot_dir, "Fig_Accuracy_Combined.html"))
fig_acc.show()

In [48]:
# ── fig_p1: Estimation Error vs. Breathing Rate ───────────────────────────────
fig_p1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["(a) Scatter & Regression",
                    "(b) Mean Error per Breathing Step"],
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.10,
)

for method in METHODS_ORDER:                         # ← FIX: uppercase
    sub   = df_err[df_err["edr_method"] == method].dropna(
                subset=["bpm_target", "error_hz"])
    color = METHOD_COLORS[method]                    # ← FIX: uppercase
    label = METHOD_LABELS[method]

    x_bpm = sub["bpm_target"].values
    y_err = sub["error_hz"].values * 60              # Hz → breaths/min

    fig_p1.add_trace(go.Scatter(
        x=x_bpm, y=y_err,
        mode="markers", name=label,
        marker=dict(color=color, size=5, opacity=0.35),
        legendgroup=label, showlegend=True,
    ), row=1, col=1)

    m, b = np.polyfit(x_bpm, y_err, 1)
    r, p = stats.pearsonr(x_bpm, y_err)
    xl   = np.linspace(x_bpm.min(), x_bpm.max(), 100)

    fig_p1.add_trace(go.Scatter(
        x=xl, y=m * xl + b, mode="lines",
        line=dict(color=color, width=2.5),
        legendgroup=label, showlegend=False,
    ), row=1, col=1)

    fig_p1.add_annotation(
        x=xl[-1], y=m * xl[-1] + b,
        text=f"  r = {r:.3f}{sig_star(p)}",
        showarrow=False, xanchor="left",
        font=dict(color=color, size=11),
        row=1, col=1,
    )

    grp   = sub.groupby("bpm_target")["error_hz"]
    means = grp.mean() * 60
    sds   = grp.std()  * 60

    fig_p1.add_trace(go.Scatter(
        x=means.index.values, y=means.values,
        mode="lines+markers", name=label,
        line=dict(color=color, width=2.5),
        marker=dict(size=9, symbol="diamond"),
        error_y=dict(type="data", array=sds.values, visible=True,
                     color=color, thickness=1.3, width=5),
        legendgroup=label, showlegend=False,
    ), row=1, col=2)

for c in [1, 2]:
    fig_p1.add_hline(y=0, line_dash="dash", line_color="gray",
                     line_width=1, row=1, col=c)

fig_p1.update_xaxes(**ax_style("Paced Breathing Rate (bpm)"),
                    tickvals=bpm_vals, ticktext=[str(int(v)) for v in bpm_vals],
                    tickangle=-45, row=1, col=1)
fig_p1.update_xaxes(**ax_style("Breathing Rate (bpm)"),
                    tickvals=bpm_vals, ticktext=[str(int(v)) for v in bpm_vals],
                    tickangle=-45, row=1, col=2)
fig_p1.update_yaxes(**ax_style("Estimation Error (breaths/min)"), row=1, col=1)
fig_p1.update_yaxes(**ax_style("Estimation Error (breaths/min)"), row=1, col=2)
fig_p1.update_layout(**paper_layout("Estimation Error by Method"))

fig_p1.write_html(os.path.join(plot_dir, "Fig_Error_vs_BreathingRate.html"))
fig_p1.show()

In [49]:
# ── fig_p2: SNR vs. Breathing Rate ───────────────────────────────────────────
fig_p2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f"(a) {METHOD_LABELS['HeartMovement']}",
                    f"(b) {METHOD_LABELS['RR_Intervals']}",
                    f"(c) {METHOD_LABELS['R_Amplitude']}"],
    shared_yaxes=True,
    horizontal_spacing=0.05,
)

for col, method in enumerate(METHODS_ORDER, 1):          # ← uppercase
    sub   = df_err[df_err["edr_method"] == method].dropna(
                subset=["bpm_target", "SNR_dB"])
    color = METHOD_COLORS[method]                        # ← uppercase
    label = METHOD_LABELS[method]
    grp   = sub.groupby("bpm_target")["SNR_dB"]
    means, sds = grp.mean(), grp.std()
    x_hz  = (means.index / 60).to_numpy()

    fig_p2.add_trace(go.Scatter(
        x=x_hz, y=means.values,
        mode="lines+markers", name=label,
        line=dict(color=color, width=2.5),
        marker=dict(color=color, size=8),
        error_y=dict(type="data", array=sds.values, visible=True,
                     color=color, thickness=1.5, width=5),
        legendgroup=label, showlegend=True,
    ), row=1, col=col)

    fig_p2.add_hline(y=0, line_dash="dash", line_color="gray",
                     line_width=1, row=1, col=col)

    # OLS-Regression über Einzelwerte
    x_raw = (sub["bpm_target"] / 60).values
    y_raw = sub["SNR_dB"].values
    m, b  = np.polyfit(x_raw, y_raw, 1)
    r, p  = stats.pearsonr(x_raw, y_raw)
    x_fit = np.linspace(x_raw.min(), x_raw.max(), 100)

    fig_p2.add_trace(go.Scatter(
        x=x_fit, y=m * x_fit + b,
        mode="lines", line=dict(color=color, width=1.8, dash="dash"),
        legendgroup=label, showlegend=False,
        hovertemplate=(f"OLS: y={m:.2f}x+{b:.2f}<br>"
                       f"r={r:.3f} {sig_star(p)}, p={p:.4f}<extra></extra>"),
    ), row=1, col=col)

    fig_p2.add_annotation(
        x=x_hz.max(), y=means.values.max() + sds.values[means.values.argmax()] * 0.5,
        text=f"r={r:.3f}{sig_star(p)}",
        showarrow=False, xanchor="right",
        font=dict(color=color, size=FS-1),
        row=1, col=col,
    )
    fig_p2.update_xaxes(**ax_style("Breathing Rate (Hz)"),
                        tickvals=hz_vals, ticktext=hz_labels,
                        tickangle=-45, row=1, col=col)

fig_p2.update_yaxes(**ax_style("SNR (dB)"), row=1, col=1)
fig_p2.update_layout(**paper_layout("Comparison of EDR Signal-to-Noise Ratios"))
fig_p2.write_html(os.path.join(plot_dir, "Fig_SNR_vs_BreathingRate.html"))
fig_p2.show()

In [50]:
# ── fig_p3: Residual Error vs. Mean RR Interval ───────────────────────────────
fig_p3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["(a) Residual vs. Mean RR Interval",
                    "(b) Residual Distribution"],
    column_widths=[0.60, 0.40],
    horizontal_spacing=0.15,
)

for method in METHODS_ORDER:                             # ← uppercase
    sub   = df_err[df_err["edr_method"] == method].dropna(
                subset=["hrv_mean_rr_ms", "Residualvarianz"])
    color = METHOD_COLORS[method]                        # ← uppercase
    label = METHOD_LABELS[method]
    x_rr, y_res = sub["hrv_mean_rr_ms"].values, sub["Residualvarianz"].values
    rgba  = f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.5)"
    r, p  = stats.pearsonr(x_rr, y_res)
    m, b  = np.polyfit(x_rr, y_res, 1)
    xl    = np.linspace(x_rr.min(), x_rr.max(), 100)

    fig_p3.add_trace(go.Scatter(
        x=x_rr, y=y_res, mode="markers", name=label,
        marker=dict(color=color, size=5, opacity=0.35),
        legendgroup=label, showlegend=True,
    ), row=1, col=1)
    fig_p3.add_trace(go.Scatter(
        x=xl, y=m*xl+b, mode="lines",
        line=dict(color=color, width=2.5),
        legendgroup=label, showlegend=False,
    ), row=1, col=1)
    fig_p3.add_annotation(
        x=x_rr.max(), y=m*x_rr.max()+b,
        text=f"  r={r:.3f}{sig_star(p)}",
        showarrow=False, xanchor="left",
        font=dict(color=color, size=11), row=1, col=1,
    )
    fig_p3.add_trace(go.Violin(
        x=[label]*len(y_res), y=y_res, name=label,
        fillcolor=rgba, opacity=0.8, line_color=color,
        box_visible=True, meanline_visible=True,
        legendgroup=label, showlegend=False,
    ), row=1, col=2)

for c in [1, 2]:
    fig_p3.add_hline(y=0, line_dash="dash", line_color="gray",
                     line_width=1, row=1, col=c)

fig_p3.update_xaxes(**ax_style("Mean RR Interval (ms)"), row=1, col=1)
fig_p3.update_xaxes(title_text="", tickangle=-20, row=1, col=2)
fig_p3.update_yaxes(**ax_style("Residual Error (Hz)"), row=1, col=1)
fig_p3.update_yaxes(**ax_style("Residual Error (Hz)"), row=1, col=2)
fig_p3.update_layout(**paper_layout(
    "Residual Estimation Error vs. Mean RR Interval", width=950, height=480))
fig_p3.write_html(os.path.join(plot_dir, "Fig_ResidualError_vs_RR.html"))
fig_p3.show()

In [51]:
# ── fig_p4: Overall Comparison ────────────────────────────────────────────────
fig_p4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["(a) Estimation Error", "(b) SNR"],
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
)

for method in METHODS_ORDER:                             # ← uppercase
    sub   = df_err[df_err["edr_method"] == method].dropna(
                subset=["error_hz", "SNR_dB"])
    color = METHOD_COLORS[method]                        # ← uppercase
    label = METHOD_LABELS[method]
    rgba  = f"rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.5)"

    for col, (yvar, ylabel) in enumerate([("error_hz", "Estimation Error (Hz)"),
                                           ("SNR_dB", "SNR (dB)")], 1):
        fig_p4.add_trace(go.Violin(
            x=[label]*len(sub), y=sub[yvar].values, name=label,
            fillcolor=rgba, opacity=0.85, line_color=color,
            box_visible=True, meanline_visible=True,
            legendgroup=label, showlegend=(col == 1),
        ), row=1, col=col)

for c in [1, 2]:
    fig_p4.add_hline(y=0, line_dash="dash", line_color="gray",
                     line_width=1, row=1, col=c)

fig_p4.update_xaxes(tickangle=-20, row=1, col=1)
fig_p4.update_xaxes(tickangle=-20, row=1, col=2)
fig_p4.update_yaxes(**ax_style("Estimation Error (Hz)"), row=1, col=1)
fig_p4.update_yaxes(**ax_style("SNR (dB)"), row=1, col=2)
fig_p4.update_layout(**paper_layout("Overall Algorithm Comparison",
                                     width=950, height=480))
fig_p4.write_html(os.path.join(plot_dir, "Fig_Overall_Comparison.html"))
fig_p4.show()


In [52]:
# ── fig_rr2: Cardiac Sampling Effect ─────────────────────────────────────────
Y_OFFSETS_ERR = {"HeartMovement":  0.040, "RR_Intervals":  0.000, "R_Amplitude": -0.040}
Y_OFFSETS_RES = {"HeartMovement":  0.040, "RR_Intervals":  0.000, "R_Amplitude": -0.040}

fig_rr2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["(a) Estimation Error vs. Mean RR Interval",
                    "(b) Residual Error vs. Mean RR Interval"],
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.13,
)

for method in METHODS_ORDER:                         # ← FIX: uppercase
    color = METHOD_COLORS[method]                    # ← FIX: uppercase
    label = METHOD_LABELS[method]

    # (a) raw error_hz
    sub_a = df_err[df_err["edr_method"] == method].dropna(
                subset=["hrv_mean_rr_ms", "error_hz"])
    x_a, y_a   = sub_a["hrv_mean_rr_ms"].values, sub_a["error_hz"].values
    r_a, p_a   = stats.pearsonr(x_a, y_a)
    m_a, b_a   = np.polyfit(x_a, y_a, 1)
    xl_a       = np.linspace(x_a.min(), x_a.max(), 100)

    fig_rr2.add_trace(go.Scatter(
        x=x_a, y=y_a, mode="markers", name=label,
        marker=dict(color=color, size=5, opacity=0.30),
        legendgroup=label, showlegend=True,
    ), row=1, col=1)
    fig_rr2.add_trace(go.Scatter(
        x=xl_a, y=m_a*xl_a+b_a, mode="lines",
        line=dict(color=color, width=2.5),
        legendgroup=label, showlegend=False,
    ), row=1, col=1)
    fig_rr2.add_annotation(
        x=xl_a[-1], y=m_a*xl_a[-1]+b_a + Y_OFFSETS_ERR[method],
        text=f"  r={r_a:.3f}{sig_star(p_a)}",
        showarrow=False, xanchor="left",
        font=dict(color=color, size=11), row=1, col=1,
    )

    # (b) Residualvarianz
    sub_b = df_err[df_err["edr_method"] == method].dropna(
                subset=["hrv_mean_rr_ms", "Residualvarianz"])
    x_b, y_b   = sub_b["hrv_mean_rr_ms"].values, sub_b["Residualvarianz"].values
    r_b, p_b   = stats.pearsonr(x_b, y_b)
    m_b, b_b   = np.polyfit(x_b, y_b, 1)
    xl_b       = np.linspace(x_b.min(), x_b.max(), 100)

    fig_rr2.add_trace(go.Scatter(
        x=x_b, y=y_b, mode="markers", name=label,
        marker=dict(color=color, size=5, opacity=0.30),
        legendgroup=label, showlegend=False,
    ), row=1, col=2)
    fig_rr2.add_trace(go.Scatter(
        x=xl_b, y=m_b*xl_b+b_b, mode="lines",
        line=dict(color=color, width=2.5),
        legendgroup=label, showlegend=False,
    ), row=1, col=2)
    fig_rr2.add_annotation(
        x=xl_b[-1], y=m_b*xl_b[-1]+b_b + Y_OFFSETS_RES[method],
        text=f"  r={r_b:.3f}{sig_star(p_b)}",
        showarrow=False, xanchor="left",
        font=dict(color=color, size=11), row=1, col=2,
    )

for c in [1, 2]:
    fig_rr2.add_hline(y=0, line_dash="dash", line_color="gray",
                      line_width=1, row=1, col=c)

fig_rr2.update_xaxes(**ax_style("Mean RR Interval (ms)"), row=1, col=1)
fig_rr2.update_xaxes(**ax_style("Mean RR Interval (ms)"), row=1, col=2)
fig_rr2.update_yaxes(**ax_style("Estimation Error (Hz)"),  row=1, col=1)
fig_rr2.update_yaxes(**ax_style("Residual Error (Hz)"),    row=1, col=2)
fig_rr2.update_layout(**paper_layout(
    "Cardiac Sampling Effect on EDR Accuracy",
    width=1200, height=460))

fig_rr2.write_html(os.path.join(plot_dir, "Fig_CardiacSampling_Effect.html"))
fig_rr2.show()

## Tabellen-Export (CSV + LaTeX-ready)

Alle statistischen Kennwerte werden in strukturierten DataFrames gesammelt
und als CSV exportiert. Die Spalten `Paper-Zitat` enthalten fertige
LaTeX-Strings die direkt in den Fließtext übernommen werden können.

| Tabelle | Inhalt |
|---|---|
| **A1** | Estimation Error: Bias, MAE, RMSE, SD, Median, IQR + Korrelationen |
| **A2** | SNR: Deskriptoren + Korrelationen mit Atemrate & Herzperiode |
| **A3** | SNR pro Methode × Breathing Step |
| **A4** | Residualfehler-Statistiken pro Methode |
| **A5** | Post-hoc Welch t-Tests zwischen Methoden-Paaren |
| **B**  | LMM Fixed Effects (lmm_E4 + lmm_res_int) |
| **B2** | Wald χ²-Tests (lmm_E4) |
| **C**  | Modell-Fit-Statistiken (LogLik, AIC, BIC) |

> **Voraussetzungen:** `df_err` mit `Residualvarianz` (Sektion 10),
> `lmm_E4`, `lmm_res_int`, `lmm_B1` (Sektionen 9–10), `METHODS_ORDER`,
> `METHOD_LABELS`, `sig_star()`.

In [53]:
# Gemeinsame Label-Map für alle LMM-Terme → Paper-Bezeichnungen
LABEL_MAP_FINAL = {
    "Intercept":                                "Intercept (HM Reference)",
    "C(edr_method)[T.RR_Intervals]":            "RR-Interval EDR vs. HM",
    "C(edr_method)[T.R_Amplitude]":             "R-Amplitude EDR vs. HM",
    "bpm_target_z":                             "Breathing Rate (z)",
    "hrv_mean_rr_ms_z":                         "Mean RR Interval (z)",
    "C(edr_method)[T.RR_Intervals]:bpm_target_z": "RR × Breathing Rate",
    "C(edr_method)[T.R_Amplitude]:bpm_target_z":  "R-Amp × Breathing Rate",
}
TERM_LABELS = {
    "Intercept":                          "Intercept (HM Reference)",
    "C(edr_method)":                      "EDR Method (main effect)",
    "bpm_target_z":                       "Breathing Rate — linear (z)",
    "C(edr_method):bpm_target_z":         "Interaction: Method × Breathing Rate",
    "hrv_mean_rr_ms_z":                   "Mean RR Interval (z)",
}
TERM_NOTES = {
    "C(edr_method)":              "df=2: 2 contrasts vs. HeartMovement",
    "bpm_target_z":               "df=1: linear predictor",
    "C(edr_method):bpm_target_z": "df=2: slope differs per method",
    "hrv_mean_rr_ms_z":           "df=1: independent of breathing rate",
}
print("✅ Label-Maps definiert.")

✅ Label-Maps definiert.


In [54]:
rows_err, rows_snr, rows_snr_step, rows_res = [], [], [], []

for method in METHODS_ORDER:                                   # ← FIX: uppercase
    sub = df_err[df_err["edr_method"] == method]

    # ── A1: Error ──────────────────────────────────────────
    sub_e = sub.dropna(subset=["error_hz"])
    err   = sub_e["error_hz"]
    t, p  = stats.ttest_1samp(err, 0)
    r_rate, p_rate = stats.pearsonr(
        sub_e.dropna(subset=["bpm_target"])["bpm_target"],
        sub_e.dropna(subset=["bpm_target"])["error_hz"])
    r_rr, p_rr = stats.pearsonr(
        sub_e.dropna(subset=["hrv_mean_rr_ms"])["hrv_mean_rr_ms"],
        sub_e.dropna(subset=["hrv_mean_rr_ms"])["error_hz"])
    rows_err.append({
        "Method":            METHOD_LABELS[method],
        "N":                 len(sub_e),
        "Bias (Hz)":         round(err.mean(), 4),
        "SD (Hz)":           round(err.std(), 4),
        "MAE (Hz)":          round(err.abs().mean(), 4),
        "RMSE (Hz)":         round(np.sqrt((err**2).mean()), 4),
        "Median (Hz)":       round(err.median(), 4),
        "IQR (Hz)":          round(err.quantile(0.75) - err.quantile(0.25), 4),
        "t (Bias≠0)":        round(t, 3),
        "p (Bias≠0)":        round(p, 4),
        "sig (Bias)":        sig_star(p),
        "r (Error|Rate)":    round(r_rate, 3),
        "p (Error|Rate)":    round(p_rate, 4),
        "sig (Rate)":        sig_star(p_rate),
        "r (Error|Mean RR)": round(r_rr, 3),
        "p (Error|Mean RR)": round(p_rr, 4),
        "sig (Mean RR)":     sig_star(p_rr),
    })

    # ── A2: SNR ────────────────────────────────────────────
    sub_s = sub.dropna(subset=["SNR_dB"])
    snr   = sub_s["SNR_dB"]
    r_rate_s, p_rate_s = stats.pearsonr(sub_s["bpm_target"],    snr)
    r_rr_s,   p_rr_s   = stats.pearsonr(sub_s["hrv_mean_rr_ms"], snr)
    rows_snr.append({
        "Method":           METHOD_LABELS[method],
        "N":                len(sub_s),
        "Mean SNR (dB)":    round(snr.mean(), 3),
        "SD SNR (dB)":      round(snr.std(), 3),
        "Median SNR (dB)":  round(snr.median(), 3),
        "IQR (dB)":         round(snr.quantile(0.75) - snr.quantile(0.25), 3),
        "Min (dB)":         round(snr.min(), 3),
        "Max (dB)":         round(snr.max(), 3),
        "r (SNR|Rate)":     round(r_rate_s, 3),
        "p (SNR|Rate)":     round(p_rate_s, 4),
        "sig (Rate)":       sig_star(p_rate_s),
        "r (SNR|Mean RR)":  round(r_rr_s, 3),
        "p (SNR|Mean RR)":  round(p_rr_s, 4),
        "sig (Mean RR)":    sig_star(p_rr_s),
    })

    # ── A3: SNR × Breathing Step ───────────────────────────
    for bpm, grp in sub.dropna(subset=["bpm_target","SNR_dB"]).groupby("bpm_target")["SNR_dB"]:
        rows_snr_step.append({
            "Method":        METHOD_LABELS[method],
            "bpm":           int(bpm),
            "Hz":            round(bpm / 60, 4),
            "Mean SNR (dB)": round(grp.mean(), 3),
            "SD (dB)":       round(grp.std(), 3),
            "Median (dB)":   round(grp.median(), 3),
            "N":             len(grp),
        })

    # ── A4: Residualfehler ─────────────────────────────────
    sub_r = sub.dropna(subset=["hrv_mean_rr_ms", "Residualvarianz"])
    res   = sub_r["Residualvarianz"]
    r_r, p_r = stats.pearsonr(sub_r["hrv_mean_rr_ms"], res)
    m_r, b_r = np.polyfit(sub_r["hrv_mean_rr_ms"].values, res.values, 1)
    rows_res.append({
        "Method":              METHOD_LABELS[method],
        "N":                   len(sub_r),
        "Mean Residual (Hz)":  round(res.mean(), 5),
        "SD Residual (Hz)":    round(res.std(), 5),
        "Median (Hz)":         round(res.median(), 5),
        "IQR (Hz)":            round(res.quantile(0.75) - res.quantile(0.25), 5),
        "r (Resid|Mean RR)":   round(r_r, 3),
        "p (Resid|Mean RR)":   round(p_r, 4),
        "sig":                 sig_star(p_r),
        "OLS Slope (Hz/ms)":   round(m_r, 6),
        "OLS Intercept":       round(b_r, 5),
    })

df_tA_err  = pd.DataFrame(rows_err).set_index("Method")
df_tA_snr  = pd.DataFrame(rows_snr).set_index("Method")
df_tA_step = pd.DataFrame(rows_snr_step)
df_tA_res  = pd.DataFrame(rows_res).set_index("Method")

SEP = "=" * 70
for title, df_t, kw in [
    ("A1 – Estimation Error",          df_tA_err,  {"index": True}),
    ("A2 – SNR",                       df_tA_snr,  {"index": True}),
    ("A3 – SNR per Breathing Step",    df_tA_step, {"index": False}),
    ("A4 – Residual Error",            df_tA_res,  {"index": True}),
]:
    print(f"\n{SEP}\nTABELLE {title}\n{SEP}")
    print(df_t.to_markdown(**kw))


TABELLE A1 – Estimation Error
| Method            |   N |   Bias (Hz) |   SD (Hz) |   MAE (Hz) |   RMSE (Hz) |   Median (Hz) |   IQR (Hz) |   t (Bias≠0) |   p (Bias≠0) | sig (Bias)   |   r (Error|Rate) |   p (Error|Rate) | sig (Rate)   |   r (Error|Mean RR) |   p (Error|Mean RR) | sig (Mean RR)   |
|:------------------|----:|------------:|----------:|-----------:|------------:|--------------:|-----------:|-------------:|-------------:|:-------------|-----------------:|-----------------:|:-------------|--------------------:|--------------------:|:----------------|
| Heartmovement EDR | 132 |     -0.0135 |    0.0411 |     0.0162 |      0.0431 |       -0.0031 |     0.0042 |       -3.779 |       0.0002 | ***          |           -0.383 |                0 | ***          |              -0.161 |              0.0648 | ns              |
| RR-Interval EDR   | 129 |     -0.0481 |    0.071  |     0.0498 |      0.0856 |       -0.004  |     0.0991 |       -7.697 |       0      | ***          |     

In [55]:
rows_lmm = []
for model_title, model in [
    ("Error ~ Breathing Rate × Method",   lmm_E4),       # ← aus Sektion 9
    ("Residual Error ~ Mean RR × Method", lmm_res_int),  # ← aus Sektion 10
]:
    ci = model.conf_int()
    for term in model.fe_params.index:
        p = model.pvalues[term]
        p_str = "< 0.001" if p < 0.001 else f"= {p:.3f}"
        rows_lmm.append({
            "Model":       model_title,
            "Term":        LABEL_MAP_FINAL.get(term, term),  # ← FIX: definiert
            "Coefficient": round(model.fe_params[term], 5),
            "SE":          round(model.bse[term], 5),
            "z":           round(model.tvalues[term], 3),
            "p":           round(p, 4),
            "CI_low":      round(ci.loc[term, 0], 5),
            "CI_high":     round(ci.loc[term, 1], 5),
            "sig":         sig_star(p),
        })

df_tB_lmm = pd.DataFrame(rows_lmm)
print(f"\n{SEP}\nTABELLE B – LMM Fixed Effects\n{SEP}")
print(df_tB_lmm.to_markdown(index=False))


TABELLE B – LMM Fixed Effects
| Model                             | Term                                           |   Coefficient |      SE |      z |      p |   CI_low |   CI_high | sig   |
|:----------------------------------|:-----------------------------------------------|--------------:|--------:|-------:|-------:|---------:|----------:|:------|
| Error ~ Breathing Rate × Method   | Intercept (HM Reference)                       |      -0.01249 | 0.00445 | -2.805 | 0.005  | -0.02122 |  -0.00376 | **    |
| Error ~ Breathing Rate × Method   | RR-Interval EDR vs. HM                         |      -0.0351  | 0.00547 | -6.417 | 0      | -0.04581 |  -0.02438 | ***   |
| Error ~ Breathing Rate × Method   | R-Amplitude EDR vs. HM                         |      -0.0164  | 0.00561 | -2.924 | 0.0035 | -0.02739 |  -0.00541 | **    |
| Error ~ Breathing Rate × Method   | Breathing Rate (z)                             |      -0.01667 | 0.00384 | -4.341 | 0      | -0.0242  |  -0.00914 | ***  

In [56]:
pairs = [
    ("HeartMovement", "RR_Intervals"),
    ("HeartMovement", "R_Amplitude"),
    ("RR_Intervals",  "R_Amplitude"),
]
rows_ph = []
for m1, m2 in pairs:
    a = df_err.loc[df_err["edr_method"] == m1, "error_hz"].dropna().values  # ← FIX: df_err
    b = df_err.loc[df_err["edr_method"] == m2, "error_hz"].dropna().values
    t, p = stats.ttest_ind(a, b, equal_var=False)
    s1, s2, n1, n2 = a.var(ddof=1), b.var(ddof=1), len(a), len(b)
    denom = (s1/n1)**2/max(n1-1,1) + (s2/n2)**2/max(n2-1,1)
    df_w  = (s1/n1 + s2/n2)**2 / denom if denom > 0 else float(n1+n2-2)
    p_str = "< 0.001" if p < 0.001 else f"= {p:.3f}"
    rows_ph.append({
        "Pair":        f"{METHOD_LABELS[m1]} vs {METHOD_LABELS[m2]}",
        "n1": n1, "n2": n2,
        "t":           round(t, 3),
        "df (Welch)":  int(round(df_w)),
        "p":           round(p, 4),
        "sig":         sig_star(p),
        "Paper-Zitat": f"$t_{{\\mathrm{{Welch}}}}({int(round(df_w))}) = {t:.2f}$, $p {p_str}$",
    })

df_ph = pd.DataFrame(rows_ph).set_index("Pair")
print(f"\n{SEP}\nTABELLE A5 – Post-hoc Welch t-Tests\n{SEP}")
print(df_ph.to_markdown())
print("\n── LaTeX-Zitate ──")
for _, row in df_ph.iterrows():
    print(" ", row["Paper-Zitat"])


TABELLE A5 – Post-hoc Welch t-Tests
| Pair                                 |   n1 |   n2 |      t |   df (Welch) |      p | sig   | Paper-Zitat                                    |
|:-------------------------------------|-----:|-----:|-------:|-------------:|-------:|:------|:-----------------------------------------------|
| Heartmovement EDR vs RR-Interval EDR |  132 |  129 |  4.807 |          204 | 0      | ***   | $t_{\mathrm{Welch}}(204) = 4.81$, $p < 0.001$  |
| Heartmovement EDR vs R-Amplitude EDR |  132 |  119 |  2.126 |          222 | 0.0346 | *     | $t_{\mathrm{Welch}}(222) = 2.13$, $p = 0.035$  |
| RR-Interval EDR vs R-Amplitude EDR   |  129 |  119 | -2.756 |          236 | 0.0063 | **    | $t_{\mathrm{Welch}}(236) = -2.76$, $p = 0.006$ |

── LaTeX-Zitate ──
  $t_{\mathrm{Welch}}(204) = 4.81$, $p < 0.001$
  $t_{\mathrm{Welch}}(222) = 2.13$, $p = 0.035$
  $t_{\mathrm{Welch}}(236) = -2.76$, $p = 0.006$


In [57]:
import warnings

def _scalar(val):
    """Robuster float-Extraktor für statsmodels wald_test Rückgaben."""
    return float(np.atleast_1d(np.asarray(val)).flat[0])

# ── B2: Wald χ²-Tests ─────────────────────────────────────────────────────────
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning,
                                module="statsmodels")
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        wt    = lmm_E4.wald_test_terms(skip_single=False)

    wt_df = wt.table.copy()
    rows_wald = []
    for term in wt_df.index:
        if term == "Intercept":
            continue
        c2  = round(_scalar(wt_df.loc[term, "statistic"]), 3)
        df_ = int(_scalar(wt_df.loc[term, "df_constraint"]))
        p   = round(_scalar(wt_df.loc[term, "pvalue"]), 4)
        p_str = "< 0.001" if p < 0.001 else f"= {p:.3f}"
        rows_wald.append({
            "Term":             TERM_LABELS.get(term, term),
            "Modell-Anmerkung": TERM_NOTES.get(term, ""),
            "χ²": c2, "df": df_, "p": p, "sig": sig_star(p),
            "Paper-Zitat": f"$\\chi^2({df_}) = {c2}$, $p {p_str}$",
        })
    df_wald = pd.DataFrame(rows_wald).set_index("Term")
    print(f"\n{SEP}\nTABELLE B2 – Wald χ²-Tests\n{SEP}")
    print(df_wald.to_markdown())
    print("\n── LaTeX-Zitate ──")
    for _, row in df_wald.iterrows():
        print(" ", row["Paper-Zitat"])
    df_wald.drop(columns=["Paper-Zitat"]).to_csv(
        os.path.join(results_dir, "TableB2_LMM_WaldChi2.csv"))

except Exception as e:
    print(f"⚠️  wald_test_terms fehlgeschlagen: {e}")
    print("→ Fallback: z² als χ²(1)")
    rows_fb = []
    for param in lmm_E4.fe_params.index:
        if param == "Intercept":
            continue
        z, p = float(lmm_E4.tvalues[param]), float(lmm_E4.pvalues[param])
        p_str = "< 0.001" if p < 0.001 else f"= {p:.3f}"
        rows_fb.append({
            "Parameter":    LABEL_MAP_FINAL.get(param, param),
            "z":            round(z, 3),
            "χ² (= z²)":   round(z**2, 3),
            "df": 1, "p":  round(p, 4), "sig": sig_star(p),
            "Paper-Zitat":  f"$z = {z:.3f}$, $p {p_str}$",
        })
    df_fb = pd.DataFrame(rows_fb).set_index("Parameter")
    print(df_fb.to_markdown())
    df_fb.drop(columns=["Paper-Zitat"]).to_csv(
        os.path.join(results_dir, "TableB2_LMM_WaldChi2_fallback.csv"))


# ── C: Modell-Fit-Statistiken ──────────────────────────────────────────────────
def model_fit_row(model, label):
    n = int(model.nobs)
    k = len(model.fe_params) + model.cov_re.shape[0] + 1

    # ← FIX: group_labels ist im Model-Objekt, nicht im Results-Objekt
    try:
        n_subj = len(model.model.group_labels)
    except AttributeError:
        n_subj = len(np.unique(model.model.groups))

    return {
        "Modell":     label,
        "N_obs":      n,
        "N_subjects": n_subj,
        "k_params":   k,
        "LogLik":     round(model.llf, 2),
        "AIC":        round(-2*model.llf + 2*k, 2),
        "BIC":        round(-2*model.llf + k*np.log(n), 2),
    }

df_fit = pd.DataFrame([
    model_fit_row(lmm_E4,      "Error ~ Rate × Method + MeanRR"),
    model_fit_row(lmm_res_int, "Residual ~ MeanRR × Method"),
]).set_index("Modell")
print(f"\n{SEP}\nTABELLE C – Modell-Fit\n{SEP}")
print(df_fit.to_markdown())
print("\n── LaTeX-Zitate ──")
for name, row in df_fit.iterrows():
    print(f"  {name}: N={row['N_obs']}, Subjects={row['N_subjects']}, "
          f"LogLik={row['LogLik']}, AIC={row['AIC']}, BIC={row['BIC']}")

# ── CSV Export ────────────────────────────────────────────────────────────────
export_map = {
    "TableA1_Error_Stats.csv":    (df_tA_err,  True),
    "TableA2_SNR_Stats.csv":      (df_tA_snr,  True),
    "TableA3_SNR_per_Step.csv":   (df_tA_step, False),
    "TableA4_Residual_Stats.csv": (df_tA_res,  True),
    "TableA5_Posthoc_Welch.csv":  (df_ph.drop(columns=["Paper-Zitat"]), True),
    "TableB_LMM_Coefficients.csv":(df_tB_lmm,  False),
    "TableC_ModelFit.csv":        (df_fit,      True),
}
for fname, (df_t, idx) in export_map.items():
    df_t.to_csv(os.path.join(results_dir, fname), index=idx)
    print(f"  💾 {fname}")
print(f"\n✅ {len(export_map)} Tabellen exportiert nach: {results_dir}")


TABELLE B2 – Wald χ²-Tests
| Term                                 | Modell-Anmerkung                    |     χ² |   df |     p | sig   | Paper-Zitat                       |
|:-------------------------------------|:------------------------------------|-------:|-----:|------:|:------|:----------------------------------|
| EDR Method (main effect)             | df=2: 2 contrasts vs. HeartMovement | 41.218 |    2 | 0     | ***   | $\chi^2(2) = 41.218$, $p < 0.001$ |
| Breathing Rate — linear (z)          | df=1: linear predictor              | 18.841 |    1 | 0     | ***   | $\chi^2(1) = 18.841$, $p < 0.001$ |
| Interaction: Method × Breathing Rate | df=2: slope differs per method      | 39.594 |    2 | 0     | ***   | $\chi^2(2) = 39.594$, $p < 0.001$ |
| Mean RR Interval (z)                 | df=1: independent of breathing rate |  5.912 |    1 | 0.015 | *     | $\chi^2(1) = 5.912$, $p = 0.015$  |

── LaTeX-Zitate ──
  $\chi^2(2) = 41.218$, $p < 0.001$
  $\chi^2(1) = 18.841$, $p < 0.001